# 03 — Model Training and Selection
## EGX Directional Price Prediction

This notebook trains and compares four model families across all 5 EGX tickers. It uses MLflow to log every experiment, then ranks the models so we can choose the strongest architecture per ticker before final evaluation.

The focus is on validation F1 and ROC-AUC, not plain accuracy, because class balance alone does not tell us whether the model is producing useful trading signals.

All splits are chronological to avoid leakage from future data into training.

## 1. Imports

This code block loads the libraries needed for model training, evaluation, plotting, and MLflow tracking. It also switches Matplotlib to a non-interactive backend so plots can be saved safely during notebook execution.

In [1]:
from __future__ import annotations

import warnings
from pathlib import Path
from typing import Any, Dict, List, Tuple

import matplotlib
matplotlib.use("Agg")  # non-interactive backend — safe for notebook artifact saving
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mlflow
from mlflow import lightgbm as mlflow_lightgbm
from mlflow import sklearn as mlflow_sklearn
from mlflow import xgboost as mlflow_xgboost

from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print("✓ All imports loaded successfully.")

d:\my_projects\Egx-analyst\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports loaded successfully.


## 2. Configuration

This block defines the fixed experiment settings: tickers, feature list, target column, date-based split anchors, random seed, output directories, and the MLflow experiment name. Keeping these values in one place makes the notebook easy to reuse and keeps the feature contract aligned with notebook 02.

In [2]:
# ── Tickers ──────────────────────────────────────────────────────────────────
TICKERS: List[str] = ["COMI_CA", "HRHO_CA", "TMGH_CA", "SWDY_CA", "ORWE_CA"]

# ── Feature contract (must match features.py / notebook 02) ──────────────────
FEATURE_COLUMNS: List[str] = [
    "RSI_14",
    "MACD_Norm", "MACD_Signal_Norm", "MACD_Hist_Norm",
    "Bollinger_Width", "Bollinger_Position",
    "ATR_Pct",
    "Return_Lag_1", "Return_Lag_5", "Return_Lag_10", "Return_Lag_21",
    "Close_MA5_Ratio", "Close_MA21_Ratio",
    "Close_CV5", "Close_CV21",
    "Volume_Ratio", "Volume_Spike",
    "Day_Of_Week", "Month", "is_Ramadan",
]
TARGET_COLUMN: str = "Target"

# ── Chronological split anchors ───────────────────────────────────────────────
# Train  : 2019-01-01 → TRAIN_END  (~70% of data, ~5 years)
# Val    : TRAIN_END  → VAL_END    (~15%, ~1 year) ← used for model selection
# Test   : VAL_END    → present    (~15%, sealed)  ← opened only in notebook 04
TRAIN_END: str = "2023-12-31"
VAL_END:   str = "2024-12-31"

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_STATE: int = 42

# ── Paths ─────────────────────────────────────────────────────────────────────
PROCESSED_DIR_CANDIDATES = [Path("../data/processed"), Path("data/processed")]
MLFLOW_DB_CANDIDATES = [Path("../mlflow.db"), Path("mlflow.db")]
ARTIFACTS_DIR = Path("mlflow_artifacts")   # local temp dir for plots before logging
ARTIFACTS_DIR.mkdir(exist_ok=True)


def resolve_existing_path(candidates: List[Path]) -> Path:
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]


PROCESSED_DIR = resolve_existing_path(PROCESSED_DIR_CANDIDATES)
MLFLOW_TRACKING_DB = resolve_existing_path(MLFLOW_DB_CANDIDATES)

# ── MLflow experiment ─────────────────────────────────────────────────────────
MLFLOW_EXPERIMENT_NAME: str = "EGX_Directional_Prediction"

print(f"Processed data directory : {PROCESSED_DIR.resolve()}")
print(f"MLflow tracking DB        : {MLFLOW_TRACKING_DB.resolve()}")
print(f"Train window             : 2019-01-01 → {TRAIN_END}")
print(f"Validation window        : {TRAIN_END} → {VAL_END}")
print(f"Test window (sealed)     : {VAL_END} → present")
print(f"Features                 : {len(FEATURE_COLUMNS)}")
print(f"MLflow experiment        : {MLFLOW_EXPERIMENT_NAME}")

Processed data directory : D:\my_projects\Egx-analyst\data\processed
MLflow tracking DB        : D:\my_projects\Egx-analyst\mlflow.db
Train window             : 2019-01-01 → 2023-12-31
Validation window        : 2023-12-31 → 2024-12-31
Test window (sealed)     : 2024-12-31 → present
Features                 : 20
MLflow experiment        : EGX_Directional_Prediction


## 3. Data Loading and Chronological Split

This section loads the engineered Parquet files and splits each ticker into train, validation, and sealed test windows by date. The split is strictly chronological, which preserves time-series integrity and prevents look-ahead leakage.

In [3]:
def load_ticker_data(ticker: str) -> pd.DataFrame:
    """Load processed parquet and enforce feature schema."""
    path = PROCESSED_DIR / f"{ticker}_features.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Parquet not found: {path}")
    df = pd.read_parquet(path)
    missing = set(FEATURE_COLUMNS + [TARGET_COLUMN]) - set(df.columns)
    if missing:
        raise ValueError(f"{ticker}: missing columns {sorted(missing)}")
    df = df.sort_index()  # guarantee chronological order
    return df


def chronological_split(
    df: pd.DataFrame,
    train_end: str,
    val_end: str,
) -> Tuple[
    pd.DataFrame, pd.Series,  # X_train, y_train
    pd.DataFrame, pd.Series,  # X_val,   y_val
    pd.DataFrame, pd.Series,  # X_test,  y_test  (sealed)
]:
    """
    Strict date-anchored split. No shuffling, no leakage.

    Returns 6 objects:
        X_train, y_train — training window
        X_val,   y_val   — validation window (for model selection)
        X_test,  y_test  — test window (sealed; returned so notebook 04 can use them)
    """
    train_mask = df.index <= train_end
    val_mask   = (df.index > train_end) & (df.index <= val_end)
    test_mask  = df.index > val_end

    def split_xy(mask):
        subset = df.loc[mask]
        return subset[FEATURE_COLUMNS].copy(), subset[TARGET_COLUMN].astype(int).copy()

    X_train, y_train = split_xy(train_mask)
    X_val,   y_val   = split_xy(val_mask)
    X_test,  y_test  = split_xy(test_mask)
    return X_train, y_train, X_val, y_val, X_test, y_test


# ── Preview splits across all tickers ────────────────────────────────────────
print(f"{'Ticker':<12} {'Train rows':>12} {'Val rows':>10} {'Test rows':>10} "
      f"{'Train UP%':>10} {'Val UP%':>8} {'Test UP%':>9}")
print("-" * 75)

for ticker in TICKERS:
    df = load_ticker_data(ticker)
    X_train, y_train, X_val, y_val, X_test, y_test = chronological_split(
        df, TRAIN_END, VAL_END
    )
    print(
        f"{ticker:<12} {len(X_train):>12,} {len(X_val):>10,} {len(X_test):>10,} "
        f"{y_train.mean()*100:>9.1f}% {y_val.mean()*100:>7.1f}% {y_test.mean()*100:>8.1f}%"
    )

Ticker         Train rows   Val rows  Test rows  Train UP%  Val UP%  Test UP%
---------------------------------------------------------------------------
COMI_CA             1,151        237        348      49.4%    48.9%     53.2%
HRHO_CA             1,151        237        348      45.9%    51.5%     48.0%
TMGH_CA             1,155        237        348      45.4%    49.4%     50.9%
SWDY_CA             1,154        237        347      46.2%    50.6%     43.8%
ORWE_CA             1,154        237        348      48.0%    48.9%     44.0%


## 4. Model Definitions

This code block defines the four candidate model families that will be compared: Logistic Regression, Random Forest, XGBoost, and LightGBM. Logistic Regression is wrapped in a scaling pipeline, while the tree-based models are left unscaled because they are naturally scale-invariant.

In [4]:
def build_model_registry() -> Dict[str, Dict[str, Any]]:
    """
    Returns a dict of {model_name: {"model": estimator, "logger": mlflow_log_fn, "params": dict}}.

    Each entry carries:
      - model   : the fitted/unfitted estimator (Pipeline or bare estimator)
      - logger  : the correct mlflow flavour logger for artifact saving
      - params  : the flat hyperparameter dict to log to MLflow
    """
    registry: Dict[str, Dict[str, Any]] = {
        "LogisticRegression": {
            "model": Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(
                    C=0.1,
                    max_iter=1000,
                    solver="lbfgs",
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                )),
            ]),
            "logger": mlflow_sklearn.log_model,
            "params": {
                "C": 0.1,
                "max_iter": 1000,
                "solver": "lbfgs",
                "class_weight": "balanced",
                "scaler": "StandardScaler",
            },
        },
        "RandomForest": {
            "model": RandomForestClassifier(
                n_estimators=300,
                max_depth=8,
                min_samples_leaf=20,
                max_features="sqrt",
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
            "logger": mlflow_sklearn.log_model,
            "params": {
                "n_estimators": 300,
                "max_depth": 8,
                "min_samples_leaf": 20,
                "max_features": "sqrt",
                "class_weight": "balanced",
            },
        },
        "XGBoost": {
            "model": XGBClassifier(
                n_estimators=400,
                max_depth=5,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=1.0,
                use_label_encoder=False,
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                verbosity=0,
            ),
            "logger": mlflow_xgboost.log_model,
            "params": {
                "n_estimators": 400,
                "max_depth": 5,
                "learning_rate": 0.05,
                "subsample": 0.8,
                "colsample_bytree": 0.8,
                "reg_alpha": 0.1,
                "reg_lambda": 1.0,
            },
        },
        "LightGBM": {
            "model": LGBMClassifier(
                n_estimators=400,
                max_depth=6,
                learning_rate=0.05,
                num_leaves=31,
                min_child_samples=20,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=1.0,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                verbose=-1,
            ),
            "logger": mlflow_lightgbm.log_model,
            "params": {
                "n_estimators": 400,
                "max_depth": 6,
                "learning_rate": 0.05,
                "num_leaves": 31,
                "min_child_samples": 20,
                "subsample": 0.8,
                "colsample_bytree": 0.8,
            },
        },
    }
    return registry


# Quick sanity print
registry_preview = build_model_registry()
print(f"Model registry contains {len(registry_preview)} architectures:")
for name in registry_preview:
    print(f"  • {name}")

Model registry contains 4 architectures:
  • LogisticRegression
  • RandomForest
  • XGBoost
  • LightGBM


## 5. Evaluation and Artifact Functions

These helper functions compute the evaluation metrics, save confusion matrices, save ROC curves, and clone fresh model instances for each run. Separating them from the training loop keeps the main loop easier to read and makes the notebook more maintainable.

In [5]:
def compute_metrics(y_true: pd.Series, y_pred: np.ndarray, y_proba: np.ndarray) -> Dict[str, float]:
    """Compute the full evaluation metric suite for one split."""
    return {
        "accuracy":  round(accuracy_score(y_true, y_pred), 4),
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall":    round(recall_score(y_true, y_pred, zero_division=0), 4),
        "f1":        round(f1_score(y_true, y_pred, zero_division=0), 4),
        "roc_auc":   round(roc_auc_score(y_true, y_proba), 4),
    }


def save_confusion_matrix_plot(
    y_true: pd.Series,
    y_pred: np.ndarray,
    model_name: str,
    ticker: str,
    split_label: str,
) -> Path:
    """Render confusion matrix and save to disk. Returns the saved path."""
    fig, ax = plt.subplots(figsize=(5, 4))
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["DOWN (0)", "UP (1)"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{ticker} | {model_name} | {split_label}", fontsize=10, fontweight="bold")
    plt.tight_layout()
    save_path = ARTIFACTS_DIR / f"{ticker}_{model_name}_cm_{split_label}.png"
    fig.savefig(save_path, dpi=120)
    plt.close(fig)
    return save_path


def save_roc_curve_plot(
    y_true: pd.Series,
    y_proba: np.ndarray,
    model_name: str,
    ticker: str,
    roc_auc: float,
) -> Path:
    """Render ROC curve for the validation split and save to disk."""
    fig, ax = plt.subplots(figsize=(5, 4))
    RocCurveDisplay.from_predictions(y_true, y_proba, ax=ax, name=model_name)
    ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random (AUC=0.50)")
    ax.set_title(f"{ticker} | {model_name} | Val ROC-AUC = {roc_auc:.4f}",
                 fontsize=10, fontweight="bold")
    ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    save_path = ARTIFACTS_DIR / f"{ticker}_{model_name}_roc.png"
    fig.savefig(save_path, dpi=120)
    plt.close(fig)
    return save_path


def clone_model(registry_entry: Dict[str, Any]) -> Any:
    """
    Return a fresh (unfitted) clone of the estimator.
    We rebuild from registry each run so experiments are fully independent.
    """
    from sklearn.base import clone as sklearn_clone
    return sklearn_clone(registry_entry["model"])


print("✓ Evaluation and artifact functions defined.")

✓ Evaluation and artifact functions defined.


## 6. MLflow Experiment Initialisation

This block connects the notebook to a local MLflow tracking store and creates the experiment if it does not already exist. All later runs will be grouped under this experiment so the results can be inspected and compared in the MLflow UI.

In [6]:
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_TRACKING_DB.resolve().as_posix()}")  # SQLite database tracking

# Get or create the experiment — idempotent
experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(MLFLOW_EXPERIMENT_NAME)
    print(f"✓ Created new MLflow experiment: '{MLFLOW_EXPERIMENT_NAME}' (id={experiment_id})")
else:
    experiment_id = experiment.experiment_id
    print(f"✓ Resuming MLflow experiment: '{MLFLOW_EXPERIMENT_NAME}' (id={experiment_id})")

print(f"  Tracking URI : {mlflow.get_tracking_uri()}")
print(f"  Experiment ID: {experiment_id}")
print()
print(f"  → Run 'mlflow ui --backend-store-uri sqlite:///{MLFLOW_TRACKING_DB.resolve().as_posix()} --port 5000' to open the dashboard.")

✓ Created new MLflow experiment: 'EGX_Directional_Prediction' (id=1)
  Tracking URI : sqlite:///D:/my_projects/Egx-analyst/mlflow.db
  Experiment ID: 1

  → Run 'mlflow ui --backend-store-uri sqlite:///D:/my_projects/Egx-analyst/mlflow.db --port 5000' to open the dashboard.


## 7. Main Training Loop

This is the core execution block. It loops through each ticker, splits the data chronologically, then trains each model architecture as a separate MLflow run while logging parameters, metrics, and artifacts. The run naming pattern makes it easy to filter results by ticker and model in the tracking UI.

In [7]:
def train_and_log(
    ticker: str,
    model_name: str,
    registry_entry: Dict[str, Any],
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
) -> Dict[str, Any]:
    """
    Train one model on one ticker, log everything to MLflow, and return
    a results dict for the cross-ticker comparison table.
    """
    run_name = f"{ticker}__{model_name}"

    with mlflow.start_run(
        experiment_id=experiment_id,
        run_name=run_name,
    ) as run:

        # ── 1. Log parameters ────────────────────────────────────────────────
        mlflow.log_param("ticker", ticker)
        mlflow.log_param("model", model_name)
        mlflow.log_param("random_state", RANDOM_STATE)
        mlflow.log_param("train_end", TRAIN_END)
        mlflow.log_param("val_end", VAL_END)
        mlflow.log_param("n_features", len(FEATURE_COLUMNS))
        mlflow.log_param("train_rows", len(X_train))
        mlflow.log_param("val_rows", len(X_val))
        mlflow.log_param("train_target_balance", round(y_train.mean(), 4))
        mlflow.log_param("val_target_balance", round(y_val.mean(), 4))
        for param_key, param_val in registry_entry["params"].items():
            mlflow.log_param(param_key, param_val)

        # ── 2. Train ─────────────────────────────────────────────────────────
        model = clone_model(registry_entry)
        model.fit(X_train, y_train)

        # ── 3. Predict on train (to detect overfitting) and val ───────────────
        def get_preds(X: pd.DataFrame):
            y_pred  = model.predict(X)
            y_proba = model.predict_proba(X)[:, 1]
            return y_pred, y_proba

        train_pred, train_proba = get_preds(X_train)
        val_pred,   val_proba   = get_preds(X_val)

        # ── 4. Compute metrics ────────────────────────────────────────────────
        train_metrics = compute_metrics(y_train, train_pred, train_proba)
        val_metrics   = compute_metrics(y_val,   val_pred,   val_proba)

        # ── 5. Log metrics (prefixed by split) ───────────────────────────────
        for metric_name, value in train_metrics.items():
            mlflow.log_metric(f"train_{metric_name}", value)
        for metric_name, value in val_metrics.items():
            mlflow.log_metric(f"val_{metric_name}", value)

        # Overfitting gap — useful for spotting models that memorise training data
        overfit_gap = round(train_metrics["roc_auc"] - val_metrics["roc_auc"], 4)
        mlflow.log_metric("overfit_gap_roc_auc", overfit_gap)

        # ── 6. Log artifacts ─────────────────────────────────────────────────
        cm_path  = save_confusion_matrix_plot(
            y_val, val_pred, model_name, ticker, "val"
        )
        roc_path = save_roc_curve_plot(
            y_val, val_proba, model_name, ticker, val_metrics["roc_auc"]
        )
        mlflow.log_artifact(str(cm_path),  artifact_path="plots")
        mlflow.log_artifact(str(roc_path), artifact_path="plots")

        # ── 7. Log model ──────────────────────────────────────────────────────
        # Use the framework-native logger so the model can be loaded with
        # mlflow.pyfunc.load_model() or the native API later.
        log_fn = registry_entry["logger"]
        log_fn(model, artifact_path="model")

        run_id = run.info.run_id

    # ── 8. Return summary row for comparison table ────────────────────────────
    return {
        "ticker":          ticker,
        "model":           model_name,
        "run_id":          run_id,
        "train_rows":      len(X_train),
        "val_rows":        len(X_val),
        "train_accuracy":  train_metrics["accuracy"],
        "train_f1":        train_metrics["f1"],
        "train_roc_auc":   train_metrics["roc_auc"],
        "val_accuracy":    val_metrics["accuracy"],
        "val_precision":   val_metrics["precision"],
        "val_recall":      val_metrics["recall"],
        "val_f1":          val_metrics["f1"],
        "val_roc_auc":     val_metrics["roc_auc"],
        "overfit_gap":     overfit_gap,
    }


print("✓ Training function defined.")

✓ Training function defined.


In [8]:
# ── Execute the full training loop ────────────────────────────────────────────
all_results: List[Dict[str, Any]] = []
total_runs = len(TICKERS) * len(build_model_registry())
run_counter = 0

for ticker in TICKERS:
    print(f"\n{'━'*60}")
    print(f"  TICKER: {ticker}")
    print(f"{'━'*60}")

    df = load_ticker_data(ticker)
    X_train, y_train, X_val, y_val, X_test, y_test = chronological_split(
        df, TRAIN_END, VAL_END
    )
    print(f"  Train: {len(X_train)} rows  |  Val: {len(X_val)} rows  |  Test (sealed): {len(X_test)} rows")

    model_registry = build_model_registry()  # fresh clones for each ticker

    for model_name, registry_entry in model_registry.items():
        run_counter += 1
        print(f"\n  [{run_counter}/{total_runs}] Training {model_name} on {ticker} ...", end=" ")

        result = train_and_log(
            ticker=ticker,
            model_name=model_name,
            registry_entry=registry_entry,
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val,
        )
        all_results.append(result)

        print(
            f"done  "
            f"val_f1={result['val_f1']:.4f}  "
            f"val_roc_auc={result['val_roc_auc']:.4f}  "
            f"overfit_gap={result['overfit_gap']:+.4f}"
        )

print(f"\n\n{'═'*60}")
print(f"  ✓ All {total_runs} runs complete.")
print(f"{'═'*60}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TICKER: COMI_CA
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Train: 1151 rows  |  Val: 237 rows  |  Test (sealed): 348 rows

  [1/20] Training LogisticRegression on COMI_CA ... 

2026/07/11 03:59:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


done  val_f1=0.4182  val_roc_auc=0.4587  overfit_gap=+0.1260

  [2/20] Training RandomForest on COMI_CA ... 

2026/07/11 03:59:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


done  val_f1=0.5322  val_roc_auc=0.5355  overfit_gap=+0.3124

  [3/20] Training XGBoost on COMI_CA ... 

...

done  val_f1=0.5044  val_roc_auc=0.5165  overfit_gap=+0.3336

  [19/20] Training XGBoost on ORWE_CA ... 

2026/07/11 04:03:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


done  val_f1=0.5408  val_roc_auc=0.5226  overfit_gap=+0.4774

  [20/20] Training LightGBM on ORWE_CA ... 

2026/07/11 04:03:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


done  val_f1=0.5446  val_roc_auc=0.5489  overfit_gap=+0.4511


════════════════════════════════════════════════════════════
  ✓ All 20 runs complete.
════════════════════════════════════════════════════════════


## 8. Cross-Ticker Results Aggregation

After training finishes, this section collects the per-run metrics into one comparison table and checks the local results against MLflow. That verification step ensures every training run was logged successfully before we choose winners.

In [9]:
# ── Build results table from local list ──────────────────────────────────────
results_df = pd.DataFrame(all_results)

# ── Verify against MLflow tracking store ─────────────────────────────────────
mlflow_runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    order_by=["metrics.val_roc_auc DESC"],
)
print(f"Local results   : {len(results_df)} rows")
print(f"MLflow runs found: {len(mlflow_runs)} rows")
if len(results_df) != len(mlflow_runs):
    print("⚠️  Count mismatch — some runs may have failed silently. Check MLflow UI.")
else:
    print("✓  Counts match — all runs logged successfully.")

# ── Full results table ────────────────────────────────────────────────────────
display_cols = [
    "ticker", "model",
    "train_f1", "train_roc_auc",
    "val_accuracy", "val_precision", "val_recall", "val_f1", "val_roc_auc",
    "overfit_gap",
]

print("\nFull results table (sorted by val_roc_auc ↓):")
display(
    results_df[display_cols]
    .sort_values("val_roc_auc", ascending=False)
    .style
    .background_gradient(subset=["val_f1", "val_roc_auc"], cmap="RdYlGn")
    .background_gradient(subset=["overfit_gap"], cmap="RdYlGn_r")  # lower overfit = greener
    .format({
        "train_f1": "{:.4f}", "train_roc_auc": "{:.4f}",
        "val_accuracy": "{:.4f}", "val_precision": "{:.4f}",
        "val_recall": "{:.4f}", "val_f1": "{:.4f}",
        "val_roc_auc": "{:.4f}", "overfit_gap": "{:+.4f}",
    })
    .set_caption("All runs — colour scale: green = better performance / lower overfitting")
)

Local results   : 20 rows
MLflow runs found: 20 rows
✓  Counts match — all runs logged successfully.

Full results table (sorted by val_roc_auc ↓):


,ticker,model,train_f1,train_roc_auc,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,overfit_gap
6,HRHO_CA,XGBoost,1.0000,1.0000,0.5359,0.5588,0.4672,0.5089,0.5496,+0.4504
19,ORWE_CA,LightGBM,0.9982,1.0000,0.5696,0.5648,0.5259,0.5446,0.5489,+0.4511
2,COMI_CA,XGBoost,1.0000,1.0000,0.5612,0.5517,0.5517,0.5517,0.5450,+0.4550
5,HRHO_CA,RandomForest,0.7675,0.8652,0.5443,0.5461,0.6803,0.6058,0.5376,+0.3276
1,COMI_CA,RandomForest,0.7371,0.8479,0.5401,0.5299,0.5345,0.5322,0.5355,+0.3124
3,COMI_CA,LightGBM,1.0000,1.0000,0.5443,0.5357,0.5172,0.5263,0.5353,+0.4647
16,ORWE_CA,LogisticRegression,0.5207,0.5756,0.5274,0.5164,0.5431,0.5294,0.5339,+0.0417
18,ORWE_CA,XGBoost,1.0000,1.0000,0.5485,0.5385,0.5431,0.5408,0.5226,+0.4774
11,TMGH_CA,LightGBM,0.9943,1.0000,0.4726,0.4574,0.3675,0.4076,0.5209,+0.4791
17,ORWE_CA,RandomForest,0.7406,0.8501,0.5274,0.5182,0.4914,0.5044,0.5165,+0.3336


## 9. Winner Selection Matrix

This block selects the best model for each ticker using validation ROC-AUC as the primary score and validation F1 as the tiebreaker. The output is the per-ticker model choice that will feed into the final evaluation notebook and the training pipeline.

In [10]:
# ── Best model per ticker ─────────────────────────────────────────────────────
best_per_ticker = (
    results_df
    .sort_values(["ticker", "val_roc_auc", "val_f1"], ascending=[True, False, False])
    .groupby("ticker", as_index=False)
    .first()
)[["ticker", "model", "val_f1", "val_roc_auc", "overfit_gap", "run_id"]]

print("Winner selection matrix (best model per ticker):")
print("=" * 80)
for _, row in best_per_ticker.iterrows():
    print(
        f"  {row['ticker']:<12}  →  {row['model']:<22}  "
        f"val_roc_auc={row['val_roc_auc']:.4f}  "
        f"val_f1={row['val_f1']:.4f}  "
        f"overfit_gap={row['overfit_gap']:+.4f}"
    )
print("=" * 80)

print("\nWinner summary table:")
display(
    best_per_ticker.drop(columns="run_id")
    .style
    .background_gradient(subset=["val_f1", "val_roc_auc"], cmap="Greens")
    .format({"val_f1": "{:.4f}", "val_roc_auc": "{:.4f}", "overfit_gap": "{:+.4f}"})
    .set_caption("Selected model per ticker based on val_roc_auc (primary) + val_f1 (tiebreaker)")
)

Winner selection matrix (best model per ticker):
  COMI_CA       →  XGBoost                 val_roc_auc=0.5450  val_f1=0.5517  overfit_gap=+0.4550
  HRHO_CA       →  XGBoost                 val_roc_auc=0.5496  val_f1=0.5089  overfit_gap=+0.4504
  ORWE_CA       →  LightGBM                val_roc_auc=0.5489  val_f1=0.5446  overfit_gap=+0.4511
  SWDY_CA       →  RandomForest            val_roc_auc=0.5133  val_f1=0.5704  overfit_gap=+0.3189
  TMGH_CA       →  LightGBM                val_roc_auc=0.5209  val_f1=0.4076  overfit_gap=+0.4791

Winner summary table:


,ticker,model,val_f1,val_roc_auc,overfit_gap
0,COMI_CA,XGBoost,0.5517,0.5450,+0.4550
1,HRHO_CA,XGBoost,0.5089,0.5496,+0.4504
2,ORWE_CA,LightGBM,0.5446,0.5489,+0.4511
3,SWDY_CA,RandomForest,0.5704,0.5133,+0.3189
4,TMGH_CA,LightGBM,0.4076,0.5209,+0.4791


## 10. Model Architecture Comparison (Averaged Across Tickers)

This section summarizes how each architecture performs on average across the five stocks. It helps answer the broader question of which model family is most reliable overall on EGX data, not just on a single ticker.

In [11]:
avg_by_model = (
    results_df
    .groupby("model")[["val_accuracy", "val_precision", "val_recall", "val_f1", "val_roc_auc", "overfit_gap"]]
    .mean()
    .round(4)
    .sort_values("val_roc_auc", ascending=False)
)

print("Average validation performance per model architecture (across 5 tickers):")
display(
    avg_by_model.style
    .background_gradient(subset=["val_f1", "val_roc_auc"], cmap="RdYlGn")
    .background_gradient(subset=["overfit_gap"], cmap="RdYlGn_r")
    .format("{:.4f}")
    .set_caption("Mean across all 5 tickers — higher ROC-AUC / F1 and lower overfit_gap = better")
)

# Bar chart of mean val_roc_auc per model
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

avg_by_model["val_roc_auc"].plot(
    kind="bar", ax=axes[0], color="steelblue", edgecolor="black", width=0.6
)
axes[0].axhline(0.5, color="red", linestyle="--", linewidth=1, label="Random baseline (0.50)")
axes[0].set_title("Mean Validation ROC-AUC by Model", fontweight="bold")
axes[0].set_ylabel("ROC-AUC")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=25, ha="right")
axes[0].legend(fontsize=8)
axes[0].set_ylim(0.45, 1.0)

avg_by_model["val_f1"].plot(
    kind="bar", ax=axes[1], color="seagreen", edgecolor="black", width=0.6
)
axes[1].set_title("Mean Validation F1-Score by Model", fontweight="bold")
axes[1].set_ylabel("F1-Score")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=25, ha="right")
axes[1].set_ylim(0.0, 1.0)

plt.suptitle("Model Architecture Comparison — Averaged Across All 5 EGX Tickers",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()

summary_chart_path = ARTIFACTS_DIR / "architecture_comparison_summary.png"
fig.savefig(summary_chart_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"Chart saved → {summary_chart_path}")

Average validation performance per model architecture (across 5 tickers):


,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,overfit_gap
model,,,,,,
LightGBM,0.5156,0.5141,0.4806,0.4956,0.5200,0.4799
XGBoost,0.5182,0.5165,0.4742,0.4936,0.5167,0.4833
RandomForest,0.5232,0.5186,0.5568,0.5345,0.5161,0.3352
LogisticRegression,0.4827,0.4801,0.5082,0.4925,0.4838,0.0932


Chart saved → mlflow_artifacts\architecture_comparison_summary.png


## 11. Per-Ticker Performance Heatmap

This code block turns the validation ROC-AUC results into a ticker-by-model heatmap. It gives a fast visual view of which model works best for each stock and where performance is consistently stronger or weaker.

In [12]:
# Pivot to ticker × model matrix
heatmap_data = results_df.pivot(index="ticker", columns="model", values="val_roc_auc")

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(heatmap_data.values, cmap="RdYlGn", aspect="auto", vmin=0.45, vmax=0.75)
plt.colorbar(im, ax=ax, label="Validation ROC-AUC")

ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_xticklabels(heatmap_data.columns, rotation=20, ha="right")
ax.set_yticklabels(heatmap_data.index)

# Annotate cells
for i in range(len(heatmap_data.index)):
    for j in range(len(heatmap_data.columns)):
        val = heatmap_data.values[i, j]
        ax.text(j, i, f"{val:.4f}", ha="center", va="center", fontsize=9,
                color="black" if 0.45 < val < 0.70 else "white")

ax.set_title("Validation ROC-AUC Heatmap — Ticker × Model", fontweight="bold", fontsize=12)
plt.tight_layout()

heatmap_path = ARTIFACTS_DIR / "roc_auc_heatmap.png"
fig.savefig(heatmap_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"Heatmap saved → {heatmap_path}")

Heatmap saved → mlflow_artifacts\roc_auc_heatmap.png


## 12. Export Winner Configuration

This block writes the selected best model per ticker into a JSON file. The downstream training code can read that file directly, so the notebook results become a reusable configuration rather than a one-off analysis.

In [13]:
import json

# Build winner config: {ticker: {model, run_id, val_roc_auc, val_f1}}
winner_config: Dict[str, Any] = {}
for _, row in best_per_ticker.iterrows():
    winner_config[row["ticker"]] = {
        "model":       row["model"],
        "run_id":      row["run_id"],
        "val_roc_auc": round(row["val_roc_auc"], 4),
        "val_f1":      round(row["val_f1"], 4),
    }

winner_path = Path("../models") if Path("../models").exists() else Path("models")
winner_path.mkdir(parents=True, exist_ok=True)
winner_file = winner_path / "winner_config.json"
winner_file.write_text(json.dumps(winner_config, indent=2))

print(f"Winner config saved → {winner_file.resolve()}")
print()
print(json.dumps(winner_config, indent=2))

Winner config saved → D:\my_projects\Egx-analyst\notebooks\models\winner_config.json

{
  "COMI_CA": {
    "model": "XGBoost",
    "run_id": "c941878a16a54ff687951f765f5b8bf8",
    "val_roc_auc": 0.545,
    "val_f1": 0.5517
  },
  "HRHO_CA": {
    "model": "XGBoost",
    "run_id": "e02f44b173634f3ea799ce01baf6fb82",
    "val_roc_auc": 0.5496,
    "val_f1": 0.5089
  },
  "ORWE_CA": {
    "model": "LightGBM",
    "run_id": "0b4f5f2e13b644d99d6a8d99d813024e",
    "val_roc_auc": 0.5489,
    "val_f1": 0.5446
  },
  "SWDY_CA": {
    "model": "RandomForest",
    "run_id": "aa24bf99cebc42b6baa3c6e06e3a934f",
    "val_roc_auc": 0.5133,
    "val_f1": 0.5704
  },
  "TMGH_CA": {
    "model": "LightGBM",
    "run_id": "cfd0ad38a2ed4024a444e9ccd87e7b5a",
    "val_roc_auc": 0.5209,
    "val_f1": 0.4076
  }
}


## 13. Summary and Next Steps

### What this notebook produced
- **20 MLflow runs** (5 tickers × 4 models), each with logged parameters, metrics, confusion matrix, ROC curve, and serialised model artifact.
- A **winner selection matrix** identifying the best architecture per ticker by val ROC-AUC.
- A `models/winner_config.json` file consumed by `src/train.py`.

### Interpreting the results
| Signal | Interpretation |
|---|---|
| val ROC-AUC > 0.55 | Meaningful signal above random; worth tuning |
| val ROC-AUC ≈ 0.50 | No predictive signal; re-examine features |
| overfit_gap > 0.10 | Model memorises training data; increase regularisation |
| val precision < 0.50 | More false UP signals than true ones; threshold adjustment needed |

### Next step — Notebook 04: `04_hyperparameter_tuning.ipynb`
Take the winning architecture per ticker and run **Optuna** hyperparameter search to find the optimal configuration before final test-set evaluation. The test set (`> 2024-12-31`) remains **sealed** until notebook 04 is complete.